PySpark DataFrame Case Study: Employee Performance Review Analysis

Step1

Purpose: Initialize the PySpark environment and create a Spark session to process and analyze employee performance review data.

In [0]:
# Creating Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Employee Performance Review Analysis") \
    .getOrCreate()

Step2

Purpose: Load the employee performance review dataset into a structured PySpark DataFrame for further analysis and transformations.

In [0]:
#Creating dataframe
data = [
 (1, '2024-01-10', 'Engineering', 5, 'John'),
 (2, '2024-01-11', 'HR', 4, 'Jane'),
 (3, '2024-01-12', 'Sales', 3, 'Sam'),
 (4, '2024-02-01', 'Engineering', 5, 'John'),
 (1, '2024-03-10', 'Engineering', 4, 'Jane'),
 (2, '2024-03-11', 'HR', None, 'Sam')
]

columns = ["emp_id", "review_date", "department", "rating", "reviewer"]

df = spark.createDataFrame(data, schema=columns)

df.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



Step3

Purpose: Identify employees with high performance ratings (4 or above) for performance analysis and reporting.

In [0]:
#Filtering operation 
#Finding ratings >= 4
df_filtered = df.filter(df.rating >= 4)

df_filtered.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
+------+-----------+-----------+------+--------+



Step4

Purpose: Replace missing ratings with the average department rating to maintain data consistency and improve data quality.

In [0]:
#Handling null values
from pyspark.sql.functions import avg, coalesce
from pyspark.sql.window import Window

window_spec = Window.partitionBy('department')

df_filled = df.withColumn(
    'rating',
    coalesce(df.rating, avg('rating').over(window_spec))
)

df_filled.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|   5.0|    John|
|     4| 2024-02-01|Engineering|   5.0|    John|
|     1| 2024-03-10|Engineering|   4.0|    Jane|
|     2| 2024-01-11|         HR|   4.0|    Jane|
|     2| 2024-03-11|         HR|   4.0|     Sam|
|     3| 2024-01-12|      Sales|   3.0|     Sam|
+------+-----------+-----------+------+--------+



Step5

Purpose: Remove duplicate employee review records to ensure accurate analysis and avoid redundant data.

In [0]:
#Removing duplicates
df_no_duplicates = df.dropDuplicates(['emp_id', 'review_date'])

df_no_duplicates.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



Step 6

Purpose: Retrieve only relevant columns required for reporting and focused analysis.

In [0]:
#Selecting specific columns
df_selected = df.select('emp_id', 'department', 'rating')

df_selected.show()

+------+-----------+------+
|emp_id| department|rating|
+------+-----------+------+
|     1|Engineering|     5|
|     2|         HR|     4|
|     3|      Sales|     3|
|     4|Engineering|     5|
|     1|Engineering|     4|
|     2|         HR|  NULL|
+------+-----------+------+



Step 7

Purpose: Calculate the average performance rating for each department to evaluate departmental performance trends.

In [0]:
#Performing GroupBy and Aggregation
df_grouped = df.groupBy('department').agg({'rating': 'avg'})

df_grouped.show()

+-----------+-----------------+
| department|      avg(rating)|
+-----------+-----------------+
|Engineering|4.666666666666667|
|         HR|              4.0|
|      Sales|              3.0|
+-----------+-----------------+



Step 8

Purpose: Combine employee review data with employee details to enrich the dataset and provide more meaningful insights.

In [0]:
#Joining Dataframes
employee_data = [
    (1, "Alex"),
    (2, "Maria"),
    (3, "David"),
    (4, "Sophia")
]

employee_columns = ["emp_id", "employee_name"]

df_employees = spark.createDataFrame(employee_data, employee_columns)

In [0]:
df_joined = df.join(df_employees, on='emp_id', how='inner')

df_joined.show()

+------+-----------+-----------+------+--------+-------------+
|emp_id|review_date| department|rating|reviewer|employee_name|
+------+-----------+-----------+------+--------+-------------+
|     1| 2024-01-10|Engineering|     5|    John|         Alex|
|     2| 2024-01-11|         HR|     4|    Jane|        Maria|
|     3| 2024-01-12|      Sales|     3|     Sam|        David|
|     4| 2024-02-01|Engineering|     5|    John|       Sophia|
|     1| 2024-03-10|Engineering|     4|    Jane|         Alex|
|     2| 2024-03-11|         HR|  NULL|     Sam|        Maria|
+------+-----------+-----------+------+--------+-------------+



Step 9

Purpose: Merge existing and newly received review datasets into a single consolidated DataFrame.

In [0]:
#Creating another reviews DataFrame
new_reviews = [
    (5, '2024-04-01', 'Marketing', 5, 'Chris')
]

df_new_reviews = spark.createDataFrame(new_reviews, columns)

In [0]:
df_union = df.union(df_new_reviews)

df_union.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|    John|
|     2| 2024-01-11|         HR|     4|    Jane|
|     3| 2024-01-12|      Sales|     3|     Sam|
|     4| 2024-02-01|Engineering|     5|    John|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
|     5| 2024-04-01|  Marketing|     5|   Chris|
+------+-----------+-----------+------+--------+



Step 11

Purpose: Enable SQL-based analysis on the DataFrame and calculate average employee ratings using Spark SQL.

In [0]:
#Creating temp view + SQL query
df.createOrReplaceTempView('performance_reviews')

sql_result = spark.sql("""
SELECT emp_id,
AVG(rating) as avg_rating
FROM performance_reviews
GROUP BY emp_id
""")

sql_result.show()

+------+----------+
|emp_id|avg_rating|
+------+----------+
|     1|       4.5|
|     2|       4.0|
|     3|       3.0|
|     4|       5.0|
+------+----------+



Step 12

Purpose: Analyze employee performance trends over time by calculating cumulative average ratings for each employee.

In [0]:
#Window Function
#Cumulative average rating over time
window_spec = Window.partitionBy('emp_id').orderBy('review_date')

df_with_cumulative_avg = df.withColumn(
    'cumulative_avg',
    avg('rating').over(window_spec)
)

df_with_cumulative_avg.show()

+------+-----------+-----------+------+--------+--------------+
|emp_id|review_date| department|rating|reviewer|cumulative_avg|
+------+-----------+-----------+------+--------+--------------+
|     1| 2024-01-10|Engineering|     5|    John|           5.0|
|     1| 2024-03-10|Engineering|     4|    Jane|           4.5|
|     2| 2024-01-11|         HR|     4|    Jane|           4.0|
|     2| 2024-03-11|         HR|  NULL|     Sam|           4.0|
|     3| 2024-01-12|      Sales|     3|     Sam|           3.0|
|     4| 2024-02-01|Engineering|     5|    John|           5.0|
+------+-----------+-----------+------+--------+--------------+

